In [ ]:
from matplotlib import pyplot as plt
import torchvision
import torchvision.transforms as transforms
import torch
from torch import nn
import numpy as np

transform = transforms.ToTensor()

criterion = nn.CrossEntropyLoss()

batch_size = 256

trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

dataiter = iter(trainloader)
images, labels = next(dataiter)

class LeakyReLUModel(nn.Module):
    def __init__(self, input_dim, n_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.batchnorm1 = nn.BatchNorm2d(32)
        self.batchnorm2 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 16 * 16, n_dim)
        self.fc2 = nn.Linear(n_dim, 512)
        self.fc3 = nn.Linear(512, 10)

        self.act = nn.LeakyReLU()

    def forward(self, x):
        x = self.act(self.batchnorm1(self.conv1(x)))
        x = self.act(self.batchnorm2(self.conv2(x)))
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)

        return x

class SigmoidModel(nn.Module):
    def __init__(self, input_dim, n_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.batchnorm1 = nn.BatchNorm2d(32)
        self.batchnorm2 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 16 * 16, n_dim)
        self.fc2 = nn.Linear(n_dim, 512)
        self.fc3 = nn.Linear(512, 10)

        self.act = nn.Sigmoid()

    def forward(self, x):
        x = self.act(self.batchnorm1(self.conv1(x)))
        x = self.act(self.batchnorm2(self.conv2(x)))
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)

        x = self.act(self.fc1(x))
        x = self.act(self.fc2(x))
        x = self.fc3(x)

        return x


class LeakyReLUDropoutModel(nn.Module):
    def __init__(self, input_dim, n_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.batchnorm1 = nn.BatchNorm2d(32)
        self.batchnorm2 = nn.BatchNorm2d(64)

        self.fc1 = nn.Linear(64 * 16 * 16, n_dim)
        self.fc2 = nn.Linear(n_dim, 512)
        self.fc3 = nn.Linear(512, 10)

        self.dropout = nn.Dropout(p=0.1)
        self.act = nn.LeakyReLU()

    def forward(self, x):
        x = self.act(self.batchnorm1(self.conv1(x)))
        x = self.act(self.batchnorm2(self.conv2(x)))
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)

        x = self.act(self.fc1(x))
        x = self.dropout(x)
        x = self.act(self.fc2(x))
        x = self.dropout(x)
        x = self.fc3(x)

        return x

def accuracy(model, dataloader):
  cnt = 0
  acc = 0

  for data in dataloader:
    inputs, labels = data
    inputs, labels = inputs.to('cuda'), labels.to('cuda')

    preds = model(inputs)
    preds = torch.argmax(preds, dim=-1)

    cnt += labels.shape[0]
    acc += (labels == preds).sum().item()

  return acc / cnt


def plot_acc(train_accs, test_accs, label1='train', label2='test'):
    x = np.arange(len(train_accs))

    plt.plot(x, train_accs, label=label1)
    plt.plot(x, test_accs, label=label2)
    plt.legend()
    plt.show()


In [ ]:


adam_train_accuracy, adam_test_accuracy = [], []
sgd_train_accuracy, sgd_test_accuracy = [], []
adam_sigmoid_train_accuracy, adam_sigmoid_test_accuracy = [], []
dropout_train_accuracy, dropout_test_accuracy = [], []

adam_leakyReLU_model = LeakyReLUModel(32 * 32 * 3, 1024)
sgd_leakyReLU_model = LeakyReLUModel(32 * 32 * 3, 1024)
adam_sigmoid_model = SigmoidModel(32 * 32 * 3, 1024)
dropout_leakyReLU_model = LeakyReLUDropoutModel(32 * 32 * 3, 1024)

from torch.optim import Adam
from torch.optim import SGD

lr = 0.001
adam_leakyReLU_model = adam_leakyReLU_model.to('cuda')
sgd_leakyReLU_model = sgd_leakyReLU_model.to('cuda')
adam_sigmoid_model = adam_sigmoid_model.to('cuda')
dropout_leakyReLU_model = dropout_leakyReLU_model.to('cuda')

adam_optimizer = Adam(adam_leakyReLU_model.parameters(), lr=lr)
sgd_optimizer = SGD(sgd_leakyReLU_model.parameters(), lr=lr)
adam_sigmoid_optimizer = Adam(adam_sigmoid_model.parameters(), lr=lr)
dropout_leakyReLU_optimizer = Adam(dropout_leakyReLU_model.parameters(), lr=lr)

n_epochs = 5

def DoModelTrain(trainloader, testloader, model, train_accuracy, test_accuracy, optimizer):
    for epoch in range(n_epochs):
        total_loss = 0.
        for data in trainloader:
            model.zero_grad()
            inputs, labels = data
            inputs, labels = inputs.to('cuda'), labels.to('cuda')

            preds = model(inputs)

            # Use CrossEntropyLoss
            loss = criterion(preds, labels)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # test
        with torch.no_grad():
            model.eval()

            # accuracy test
            train_accuracy.append(accuracy(model, trainloader))
            test_accuracy.append(accuracy(model, testloader))

            model.train()

        print(f"Epoch {epoch:3d} | Loss: {total_loss}")

print("Adam LeakyReLU Start")
DoModelTrain(trainloader, testloader, adam_leakyReLU_model, adam_train_accuracy, adam_test_accuracy, adam_optimizer)
print("SGD LeakyReLU Start")
DoModelTrain(trainloader, testloader, sgd_leakyReLU_model, sgd_train_accuracy, sgd_test_accuracy, sgd_optimizer)
print("Adam Sigmoid Start")
DoModelTrain(trainloader, testloader, adam_sigmoid_model, adam_sigmoid_train_accuracy, adam_sigmoid_test_accuracy, adam_sigmoid_optimizer)
print("Adam LeakyReLU Dropout Start")
DoModelTrain(trainloader, testloader, dropout_leakyReLU_model, dropout_train_accuracy, dropout_test_accuracy, dropout_leakyReLU_optimizer)

plot_acc(adam_train_accuracy, sgd_train_accuracy, "adam_train", "sgd_train")
plot_acc(adam_train_accuracy, adam_sigmoid_train_accuracy, "LeakyReLU_train", "Sigmoid_train")

x = np.arange(len(adam_train_accuracy))
plt.plot(x, adam_train_accuracy, label="LeakyReLU_train")
plt.plot(x, dropout_train_accuracy, label="Dropout_train")
plt.plot(x, adam_test_accuracy, label="LeakyReLU_test")
plt.plot(x, dropout_test_accuracy, label="Dropout_test")
plt.legend()
plt.show()